# FellaRide Propensity Scorer: training and inference

**Checkpoint 1 notebook.** Trains the four propensity models (driver, rider, connector, referrer) used by the Intervention Engine and shows inference on sample members.

**Data:** all data here is **synthetic** (labelled per hackathon rule 3). Synthetic communities are replayed through the real engine; every nudge outcome becomes one training row:
`features at the time of the nudge → responded (1) or ignored (0)`.

Features (same order the API uses): `has_car, corridor_density, recent_rides, social_degree, past_response, bias`.

In [1]:
import sys, json, pickle
from pathlib import Path
BACKEND = Path.cwd().parent
sys.path.insert(0, str(BACKEND))
from ml.train_scorer import generate_logs, train, save, MIN_SAMPLES, MIN_AUC
from app.engine.context import FEATURES
print("features:", FEATURES)

features: ['has_car', 'corridor_density', 'recent_rides', 'social_degree', 'past_response', 'bias']


## 1. Generate synthetic training logs
Half the runs use the smart engine, half use random nudges, so the data covers members the smart engine would not pick.

In [2]:
logs = generate_logs(runs=6, days=30)
for role, rows in logs.items():
    pos = sum(y for _, y in rows)
    print(f"{role:10s} samples={len(rows):5d}  positives={pos:4d}  rate={pos/len(rows):.2f}")

driver     samples=  554  positives= 147  rate=0.27
rider      samples= 1914  positives= 563  rate=0.29
connector  samples=   52  positives=  12  rate=0.23
referrer   samples=  306  positives=  37  rate=0.12


## 2. Train logistic regression per role
A model ships only if it has at least the minimum samples and beats the held-out AUC bar; otherwise the engine keeps its default weights for that role.

In [3]:
models, report = train(logs)
print(f"quality bar: samples >= {MIN_SAMPLES}, holdout AUC >= {MIN_AUC}\n")
for role, r in report.items():
    print(f"{role:10s} shipped={r.get('shipped')}  samples={r.get('samples')}  holdout_auc={r.get('holdout_auc')}")
    if 'weights' in r: print('           weights:', r['weights'], ' intercept:', r['intercept'])

quality bar: samples >= 100, holdout AUC >= 0.6

driver     shipped=True  samples=554  holdout_auc=0.759
           weights: {'has_car': -0.0, 'corridor_density': -0.039, 'recent_rides': 0.562, 'social_degree': -0.415, 'past_response': 1.995, 'bias': -0.0}  intercept: -1.758
rider      shipped=True  samples=1914  holdout_auc=0.654
           weights: {'has_car': 0.0, 'corridor_density': 0.498, 'recent_rides': 0.84, 'social_degree': 0.459, 'past_response': 1.003, 'bias': -0.004}  intercept: -1.948
connector  shipped=False  samples=52  holdout_auc=0.795
           weights: {'has_car': 0.238, 'corridor_density': 0.796, 'recent_rides': 0.0, 'social_degree': -0.252, 'past_response': 0.745, 'bias': 0.0}  intercept: -1.391
referrer   shipped=True  samples=306  holdout_auc=0.671
           weights: {'has_car': -0.269, 'corridor_density': 0.299, 'recent_rides': 0.733, 'social_degree': 0.261, 'past_response': 0.445, 'bias': 0.0}  intercept: -2.375


## 3. Save the model artifact
`models/scorer.pkl` is the file the deployed API loads at startup (`app/engine/scorer.py`).

In [4]:
out = BACKEND / "models" / "scorer.pkl"
save(models, report, out)
print("saved", out, out.stat().st_size, "bytes")

saved /Users/chirayusaxena/Downloads/fellaride 3/backend/models/scorer.pkl 2153 bytes


## 4. Inference on sample members
Create a fresh synthetic community, run 10 days through the engine, then score a few members with the saved model (exactly as `GET /api/v1/tenants/{id}/members/{id}` does).

In [5]:
from sqlalchemy.orm import sessionmaker
from sqlalchemy import select
from app.db import make_engine, init_db
from app.sim.synthetic import create_synthetic_tenant
from app.sim.behaviour import simulate
from app.engine.context import build_context
from app.engine import scorer
from app.models import Member

scorer.load_pretrained.cache_clear()
eng = make_engine("sqlite://"); init_db(eng)
db = sessionmaker(bind=eng, expire_on_commit=False)()
t = create_synthetic_tenant(db, name="inference demo", seed=999); db.commit()
simulate(db, t, 10, seed=5)
ctx = build_context(db, t)
w = scorer.tenant_weights(db, t)
sample = [m for m in ctx.members.values() if m.registered_day is not None][:8]
print(f"{'member':14s} {'car':>3s} {'state':11s} " + " ".join(f"{r:>9s}" for r in scorer.ROLES))
for m in sample:
    s = scorer.score_member(w, ctx.features(m))
    print(f"{m.name:14s} {'yes' if m.has_car else 'no':>3s} {m.state:11s} " + " ".join(f"{s[r]:9.2f}" for r in scorer.ROLES))

member         car state          driver     rider connector  referrer
Saanvi B.       no referrer         0.19      0.38      0.27      0.12
Kavya J.       yes registered       0.33      0.39      0.18      0.09
Riya P.        yes registered       0.14      0.25      0.19      0.08
Varun V.        no registered       0.31      0.39      0.20      0.12
Pooja P.        no registered       0.30      0.40      0.21      0.12
Harsh B.        no registered       0.11      0.28      0.25      0.11
Divya A.        no registered       0.32      0.38      0.20      0.12
Manav V.       yes registered       0.30      0.42      0.21      0.10


## 5. Sanity check: smart engine vs random nudges (20 days)
The scorer only matters if it changes outcomes. Same synthetic community, two strategies.

In [6]:
from app.engine import metrics
for strategy in ("smart", "random"):
    d = sessionmaker(bind=eng, expire_on_commit=False)()
    tt = create_synthetic_tenant(d, name=strategy, seed=42); d.commit()
    simulate(d, tt, 20, strategy=strategy, seed=3)
    k = metrics.kpis(d, tt)
    print(f"{strategy:6s} active={k['active_users']:3d} rides={k['rides_completed']:4d} match_rate={k['match_rate']:.2f} registered={k['registered']}")

smart  active= 33 rides=  86 match_rate=0.86 registered=76


random active= 13 rides=  14 match_rate=0.35 registered=56


**Notes.** Labels and behaviour are synthetic, so AUC values show the pipeline works; they are not evidence about real users. In production the same script retrains on real `interventions` rows, and the API also updates weights online after every outcome (`scorer.learn`).